# P5 r2 — 대용량 결과 회수 / 로컬 CPU 전환

**GPU 추출·probe를 다시 실행하지 않습니다.** 기존 Drive 폴더에서 파일을 읽어 약 512MiB 이하의 독립 ZIP으로 나눕니다(단일 원본 파일이 더 크면 해당 ZIP은 예외). 실험 config·cache·계수는 변경하지 않습니다. 작은 metadata ZIP부터 전달하면 진행 상태를 확인할 수 있습니다. 로컬 probe에는 metadata뿐 아니라 해당 seed의 trained/init cache 전체가 필요합니다. 총 전송량은 줄지 않지만 한 파일의 다운로드 부담을 줄입니다.

기존 probe 셀이 돌고 있다면 셀을 중단한 뒤 실행하세요. 이미 저장된 probe JSON은 재사용할 수 있고, 저장 중이던 한 작업은 다시 계산할 수 있습니다. Colab 런타임이 종료됐어도 Drive에 저장된 파일로 회수할 수 있습니다. 이 노트북은 CPU 런타임으로 충분합니다. 입력 번들을 다시 업로드하거나 패키지를 설치할 필요가 없습니다.

In [ ]:
from google.colab import drive, files
from pathlib import Path
from datetime import datetime, timezone
import json, sys, subprocess
drive.mount('/content/drive')
RUN=Path('/content/drive/MyDrive/boolean_interp_v1_4/P5_r2')
assert (RUN/'contract.json').exists(), '기존 P5_r2 Drive 폴더를 확인하세요.'
EXPORT=RUN.parent/('P5_transfer_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S'))
print('원본:',RUN,'분할 ZIP 저장:',EXPORT)

In [ ]:
helper=Path('/content/p5_transfer.py')
helper.write_text('"""Transport-only P5 multipart export/import; frozen experiment code stays unchanged."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport zipfile\n\n\ndef sha(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for b in iter(lambda:f.read(1024*1024),b\'\'):h.update(b)\n    return h.hexdigest()\n\n\ndef pack(root,destination,seed=None,limit=512*1024**2):\n    root=Path(root);destination=Path(destination)\n    if destination.resolve().is_relative_to(root.resolve()):raise ValueError(\'Export folder must be outside run folder\')\n    destination.mkdir(parents=True,exist_ok=True)\n    contract=root/\'contract.json\'\n    if not contract.exists():raise ValueError(\'Missing run contract; select P5_r2 folder\')\n    files=[]\n    for p in sorted(root.rglob(\'*\')):\n        if not p.is_file() or p.name.endswith(\'.tmp\'):continue\n        name=p.relative_to(root)\n        is_array=name.parts[0]==\'cache\' and p.suffix==\'.npz\'\n        if seed is None:\n            if not is_array:files.append(p)\n        elif is_array and name.parts[1] in (f\'seed{seed}_trained\',f\'seed{seed}_init\'):\n            marker=p.with_suffix(\'.json\')\n            if not marker.exists():continue\n            info=json.loads(marker.read_text())\n            if sha(p)!=info[\'sha256\']:raise ValueError(\'Corrupt source cache: \'+str(p))\n            files.extend([p,marker])\n    if not files:raise ValueError(\'No completed files for requested scope\')\n    # Each part is an independently usable ZIP, not a byte slice of a 15GB archive.\n    groups=[];current=[];size=0\n    for p in files:\n        n=p.stat().st_size\n        if current and size+n>limit:groups.append(current);current=[];size=0\n        current.append(p);size+=n\n    if current:groups.append(current)\n    scope=\'metadata\' if seed is None else f\'seed{seed}\'\n    index=dict(schema=\'p5-transfer-v1\',scope=scope,contract_sha256=sha(contract),parts=[])\n    for i,group in enumerate(groups):\n        path=destination/f\'p5_{scope}_{i+1:03d}.zip\'\n        inventory={str(p.relative_to(root)):sha(p) for p in group}\n        manifest=dict(schema=\'p5-transfer-part-v1\',contract_sha256=sha(contract),scope=scope,files=inventory)\n        if path.exists():\n            with zipfile.ZipFile(path) as z:\n                if json.loads(z.read(\'transfer_manifest.json\'))!=manifest:raise ValueError(\'Existing part differs; use new export folder\')\n                for name,digest in inventory.items():\n                    h=hashlib.sha256()\n                    with z.open(name) as f:\n                        for b in iter(lambda:f.read(1024*1024),b\'\'):h.update(b)\n                    if h.hexdigest()!=digest:raise ValueError(\'Corrupt existing part\')\n        else:\n            tmp=path.with_suffix(\'.zip.tmp\')\n            with zipfile.ZipFile(tmp,\'w\',zipfile.ZIP_DEFLATED,compresslevel=1) as z:\n                for p in group:z.write(p,str(p.relative_to(root)))\n                z.writestr(\'transfer_manifest.json\',json.dumps(manifest))\n            tmp.replace(path)\n        index[\'parts\'].append(dict(name=path.name,bytes=path.stat().st_size,sha256=sha(path),files=len(group)))\n        print(path.name,path.stat().st_size,flush=True)\n    target=destination/f\'p5_{scope}_index.json\'\n    content=json.dumps(index,indent=2)+\'\\n\'\n    if target.exists() and target.read_text()!=content:raise ValueError(\'Index conflict\')\n    target.write_text(content)\n    return index\n\n\ndef receive(archive,output,expected_contract):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    with zipfile.ZipFile(archive) as z:\n        names=z.namelist();manifest=json.loads(z.read(\'transfer_manifest.json\'))\n        if manifest[\'contract_sha256\']!=expected_contract:raise ValueError(\'Different experiment contract\')\n        if len(names)!=len(set(names)) or set(names)!=set(manifest[\'files\'])|{\'transfer_manifest.json\'}:raise ValueError(\'Archive inventory mismatch\')\n        for name,digest in manifest[\'files\'].items():\n            dest=output/name\n            if not dest.resolve().is_relative_to(output.resolve()):raise ValueError(\'Unsafe archive path\')\n            if dest.exists():\n                if sha(dest)!=digest:raise ValueError(\'Existing file conflict: \'+name)\n                continue\n            dest.parent.mkdir(parents=True,exist_ok=True)\n            tmp=dest.with_suffix(dest.suffix+\'.transfer.tmp\')\n            try:\n                with z.open(name) as src,tmp.open(\'wb\') as dst:shutil.copyfileobj(src,dst,1024*1024)\n                if sha(tmp)!=digest:raise ValueError(\'Checksum failure: \'+name)\n                tmp.replace(dest)\n            finally:\n                if tmp.exists():tmp.unlink()\n    return len(manifest[\'files\'])\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();sub=p.add_subparsers(dest=\'action\',required=True)\n    a=sub.add_parser(\'pack\');a.add_argument(\'root\');a.add_argument(\'destination\');a.add_argument(\'--seed\',type=int,choices=[0,1,2])\n    a=sub.add_parser(\'receive\');a.add_argument(\'archive\');a.add_argument(\'output\');a.add_argument(\'--contract\',default=\'experiment_v1_4/p5_r2/contract.json\')\n    args=p.parse_args()\n    if args.action==\'pack\':pack(args.root,args.destination,args.seed)\n    else:print(\'Verified imported files:\',receive(args.archive,args.output,sha(args.contract)))\n')
namespace={'__name__':'p5_transfer'}
exec(compile(helper.read_text(),str(helper),'exec'),namespace)
pack=namespace['pack']

## 1. 작은 진행 증빙부터 회수
cache 배열을 제외한 계약·환경·checksum·라벨·완료 probe 결과를 묶습니다. 이 ZIP과 index를 먼저 내려받아 전달하세요. 현재 진행 정도에 따라 여러 조각일 수 있습니다. cache 검증을 완료했다는 의미는 아닙니다.

In [ ]:
metadata=pack(RUN,EXPORT)
print(json.dumps(metadata,indent=2))
files.download(str(EXPORT/'p5_metadata_index.json'))

In [ ]:
PART=1 # metadata index의 번호를 선택합니다. 여러 조각이면 하나씩 반복하세요.
files.download(str(EXPORT/metadata['parts'][PART-1]['name']))

## 2. 로컬 CPU 실행용 cache 회수
먼저 seed 0만 선택합니다. 약 5.9GB raw cache가 여러 ZIP으로 나뉩니다. 다음 seed는 로컬 저장 공간을 확인한 뒤 회수하세요. Drive에는 선택한 seed ZIP 용량만큼 추가 여유 공간이 필요합니다. metadata만으로 로컬 probe를 실행할 수는 없습니다.

In [ ]:
SEED=0 # 0, 1, 2 중 하나
cache=pack(RUN,EXPORT,seed=SEED)
files.download(str(EXPORT/f'p5_seed{SEED}_index.json'))
print(json.dumps(cache,indent=2))

In [ ]:
PART=1 # 1부터 마지막 part까지 한 번에 하나씩 다운로드하세요.
files.download(str(EXPORT/cache['parts'][PART-1]['name']))

## 3. 다운로드가 또 실패하면
위 EXPORT 경로의 조각 ZIP은 Drive에도 보존됩니다. Drive 웹에서 해당 ZIP **하나씩** 다운로드하세요. 폴더 전체 다운로드는 다시 큰 ZIP으로 묶일 수 있으므로 피하세요. 이미 받은 조각을 다시 만들거나 GPU 추출을 반복할 필요는 없습니다.

## 4. 로컬 처리
작은 metadata ZIP과 index를 먼저 Codex에 전달하세요. 로컬 도구 `scripts/p5_transfer.py receive`가 각 독립 ZIP의 contract·파일 checksum을 검증하여 공통 폴더로 가져옵니다. 선택 seed의 모든 cache 조각을 받은 뒤 기존 `interp_v1_4.p5 probes --seeds 0` 실행기로 이어갑니다. 완료 probe JSON은 건너뛰고 새 CPU 환경 ID를 기록합니다.

로컬 공간이 부족하면 외장 SSD 또는 seed별 순차 작업이 필요합니다. cache 원본은 P6 이후에도 쓰므로 Drive에서 삭제하지 마세요. 이 회수 절차만으로 P5를 완료 처리하지 않습니다.